<a href="https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's search and engagement performance, for one client, on one calendar day — from fact_content_daily_performance, filtered to month=2026-03. Verified below: the grain check confirms 9,841,378 total rows and 9,841,378 distinct (report_date, client_hash_id, content_hash_id) combinations — an exact match. The date span runs 2026-03-01 to 2026-03-31, a full mid-panel month, deliberately not the sealed final month.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"


In [21]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label: gsc_avg_position — the ranking signal this lane analyzes.

Features (5, used honestly): gsc_impressions, ga4_engaged_sessions, sessions_organic, sessions_ai, scroll_events — same-day signals that don't algebraically depend on the label.

Context (kept for reference, not modeled): report_date, client_has_gsc, client_has_ga4, month.

Excluded: client_hash_id and content_hash_id — identifiers, not signals; including them risks the model learning an ID rather than a real pattern, and the Data Use Terms forbid treating hash keys as meaningful content. Also excluded from the honest feature set: gsc_sum_position and gsc_clicks — both are algebraically part of how gsc_avg_position is computed, so including either would leak the label (demonstrated deliberately in Section 3).

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sanity check: confirm every field named above actually exists in the table
cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0").df()['column_name'].tolist()

named_fields = ['gsc_avg_position', 'gsc_impressions', 'ga4_engaged_sessions', 'sessions_organic',
                'sessions_ai', 'scroll_events', 'report_date', 'client_has_gsc', 'client_has_ga4',
                'month', 'client_hash_id', 'content_hash_id', 'gsc_sum_position', 'gsc_clicks']

missing = [f for f in named_fields if f not in cols]
print("All fields found" if not missing else f"Missing: {missing}")

All fields found


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Deliberately including gsc_sum_position (algebraically part of the label) raised R² from 0.0033 to 0.0624 — roughly a 19x increase. This confirms real leakage, but the jump is far smaller than "near-perfect" because gsc_avg_position is a ratio (gsc_sum_position / gsc_impressions), and LinearRegression can only combine features additively — it can't represent division between its own inputs. The leak is real; a model class capable of learning ratios (e.g. a tree-based model) would likely expose it far more dramatically. Either way, the honest number for this lane is R² ≈ 0.003 — a reminder that even a "clean" feature set doesn't yet explain much of gsc_avg_position's variance.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┐
│ total_rows │ distinct_grain │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │        9841378 │
└────────────┴────────────────┘

In [24]:
con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

┌───────────┬────────────┬────────────┐
│ row_count │ first_day  │  last_day  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [25]:
con.sql(f"""
    SELECT COUNT(*) AS available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

In [26]:
# Five-feature frame for this lane, plus the deliberate leak

df = con.sql(f"""
    SELECT gsc_avg_position, gsc_impressions, ga4_engaged_sessions,
           sessions_organic, sessions_ai, scroll_events,
           gsc_sum_position  -- deliberate leak: algebraically part of the label
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

print("Rows before dropna:", len(df))
df = df.dropna()
print("Rows after dropna:", len(df))

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# WITH the leak
X_leak = df[['gsc_impressions','ga4_engaged_sessions','sessions_organic','sessions_ai','scroll_events','gsc_sum_position']]
y = df['gsc_avg_position']
X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.2, random_state=42)
model_leak = LinearRegression().fit(X_train, y_train)
print("WITH leak, R²:", model_leak.score(X_test, y_test))

# WITHOUT the leak — the honest number
X_honest = df[['gsc_impressions','ga4_engaged_sessions','sessions_organic','sessions_ai','scroll_events']]
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model_honest = LinearRegression().fit(X_train, y_train)
print("WITHOUT leak, R²:", model_honest.score(X_test, y_test))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows before dropna: 3611061
Rows after dropna: 2082695
WITH leak, R²: 0.062428722748663734
WITHOUT leak, R²: 0.0032607032276461556


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Filtering gsc_data_available IS TRUE leaves 3,611,061 of 9,841,378 rows (about 36.7%) — nearly two-thirds of this month's rows have no usable GSC signal. Per the dataset card, this is an unbalanced panel: per-client history depth differs, so unavailability isn't random — it systematically skews toward clients whose GSC history starts later than March 2026. Any ranking-signal conclusion from this slice should be read as observed and decision-support, describing only the ~37% of client-days where GSC data exists.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY gsc_data_available
""")

┌────────────────────┬─────────┐
│ gsc_data_available │    n    │
│      boolean       │  int64  │
├────────────────────┼─────────┤
│ false              │ 6230317 │
│ true               │ 3611061 │
└────────────────────┴─────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.